In [ ]:
# Goal is to take a flat 2D representation of panel data with time
# and convert to a 3D representation, where time is the 3rd dimension

import numpy as  np
import torch 

In [ ]:
bikes_numpy = np.loadtxt(
    '../../data/p1ch4/bike-sharing-dataset/hour-fixed.csv',
    dtype = np.float32,
    delimiter=',',
    skiprows = 1,
    converters = {1: lambda x: float(x[8:10])} # converts date strings to numbers
)

bikes = torch.from_numpy(bikes_numpy)
bikes

In [ ]:
bikes.shape, bikes.stride()

In [ ]:
# each row is one hour, so 17520 hours with 17 data types recorded per hour
# we are going to reshape to have a day axis, an hour axis, and then our
# recorded values axis

# -1 is the "whatever remains" value for the dimension
# this is why we typically keep the most known structured data
# at the end of the tensor, i.e. we know there are 17 values
# and then we know we want to keep them in batches of 24 hours
# finally (or first dimension) is whatever is left over, i.e.
# the number of days (17520/24)
daily_bikes = bikes.view(-1,24, bikes.shape[1]) 
daily_bikes.shape, daily_bikes.stride()

In [ ]:
17520/24

It may help to think of the tensor structure as follows:

The final value on the shape attribute is the number of columns. This really is saying that our primative data point is typically row vector, so the shape primarily comes from how many dimensions are in this row vector. 

As we add more data points, we get more rows, hence the second value: the number of rows. 

However, with more general data sets, we want to make it such that the most primitive data is first a row vector and then multiple instances of the row vectors build up to the 2d matrix. Further dimensions yield multiple "faces" of the 2d matrix and then beyond that.

For this example, the end goal is to have columns representing hours of the day, kind of like reading time left to right. As we do so, we get the 17 data values for each row. So, really, our primitive is how any data value changes with the hour. Finally, we stack a day's worth of hours & values (a 2d matrix) into batches of days, the third dimension.

In [ ]:
# finally we rearrange to get N sequences of C channels (values) for L hours in a day
daily_bikes = daily_bikes.transpose(1,2)
daily_bikes.shape, daily_bikes.stride()

408 indicates that in memory, where the data is stored sequentially, to get from one day to the next, we need to move 408 = 17x24 data points. To move sequentially in values, we need only move 1 value. Finally, to jump to the same data value but one hour ahead, we need to jump 17 places.

This is all just memory indexing, so out of sight for us, but it does indicate that these manipulations are incredibly efficient.

In [ ]:
# weather is categorical data with numerical values 1, 2, 3, & 4
# we are going to convert theses to one-hot-encoded values

# limit to first day for illustrative purposes
first_day = bikes[:24].long() 
weather_onehot = torch.zeros(first_day.shape[0],4)
first_day[:,9] # weather category is in 9th column

In [ ]:
weather_onehot.scatter_(
    dim=1, # across the columns
    index = first_day[:,9].unsqueeze(1).long() - 1, # all rows with category labels now 0,1,2, & 3
    value = 1.0
)

In [ ]:
# now concatenate this variable to our original data set
torch.cat((bikes[:24], weather_onehot),1)[:1] # across the columns, i.e. line up as you'd think with extra columns
# we are showing the first row

In [ ]:
# let's go back to daily bikes
daily_bikes.shape

In [ ]:
# we want to make a set of zeros with the same shape as the first and last dimension and 4 new variables in the middle
daily_weather_onehot = torch.zeros(daily_bikes.shape[0], 4, daily_bikes.shape[-1])
daily_weather_onehot.shape

In [ ]:
daily_bikes.shape

In [ ]:
daily_bikes[:,9,:].shape

In [ ]:
# now we scatter the one-hot-encoding along the 4 variables
daily_weather_onehot.scatter_(
    dim = 1,
    index = daily_bikes[:,9,:].unsqueeze(1).long() - 1,
    value = 1.0
)

In [ ]:
daily_weather_onehot.shape

In [ ]:
daily_bikes = torch.cat((daily_bikes, daily_weather_onehot),dim=1)

In [ ]:
# we still have to rescale the variables, e.g. temperature to 0,1 or -1,1
temp = daily_bikes[:,10,:]
temp_min = torch.min(temp)
temp_max = torch.max(temp)
daily_bikes[:,9,:] = (temp - temp_min)/(temp_max - temp_min)

In [ ]:
# alternatively, we can rescale with mean and std
temp = daily_bikes[:,10,:]
temp_mean = torch.mean(temp)
temp_std = torch.std(temp)
daily_bikes[:,9,:] = (temp - temp_mean)/ temp_std